In [5]:
import pandas as pd
import numpy as np
from scipy import stats
import os

In [2]:
raw_path = '../data/raw/'

# Dictionary to store all dataframes
dfs = {}

# Load each CSV file into the dictionary
for file in os.listdir(raw_path):
    if file.endswith('.csv'):
        df_name = file.replace('.csv', '')
        dfs[df_name] = pd.read_csv(os.path.join(raw_path, file))
        print(f"Loaded: {file}")

Loaded: Crud_Oil_Price.csv
Loaded: uk_cpih_fuel_energy_inflation.csv
Loaded: uk_cpih_inflation_by_category.csv
Loaded: uk_cpih_inflation_with_yoy.csv
Loaded: uk_fuel_summary_by_income_group.csv
Loaded: uk_fuel_transport_spending_by_decile.csv
Loaded: uk_household_expenditure_by_decile.csv
Loaded: uk_household_spending_by_income_group.csv
Loaded: uk_petrol_diesel.csv
Loaded: uk_petrol_diesel_spending_by_decile.csv


In [3]:
def clean_dataframe(df, df_name):
    """
    Clean a dataframe by:
    1. Converting date columns to datetime
    2. Checking and handling missing values
    3. Removing duplicates
    4. Detecting and handling outliers using IQR method
    """
    
    print(f"\n{'='*60}")
    print(f"Processing: {df_name}")
    print(f"{'='*60}")
    
    # Make a copy to avoid modifying original
    df_clean = df.copy()
    
    #  Convert date columns to datetime
    date_columns = [col for col in df_clean.columns if 'Date' in col or 'Time' in col or col == 'mmm-yy']
    
    for col in date_columns:
        try:
            df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
            print(f"✓ Converted {col} to datetime")
        except:
            print(f"⚠ Could not convert {col} to datetime")
    
    # MISSING VALUES - Check for null values
    missing_counts = df_clean.isnull().sum()
    missing_cols = missing_counts[missing_counts > 0]
    
    if len(missing_cols) > 0:
        print(f"\nMissing Values Found:")
        print(missing_cols)
        
        # Handle missing values
        for col in missing_cols.index:
            if df_clean[col].dtype in ['float64', 'int64']:
                # Fill numerical missing values with median
                df_clean[col].fillna(df_clean[col].median(), inplace=True)
                print(f"✓ Filled missing values in {col} with median")
            else:
                # Fill categorical with mode
                df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
                print(f"✓ Filled missing values in {col} with mode")
    else:
        print("\n✓ No missing values found")
    
    # 3. DUPLICATES - Check for and remove duplicate rows
    duplicate_count = df_clean.duplicated().sum()
    if duplicate_count > 0:
        df_clean.drop_duplicates(inplace=True)
        print(f"\n✓ Removed {duplicate_count} duplicate rows")
    else:
        print(f"\n✓ No duplicate rows found")
    
    # OUTLIER DETECTION & HANDLING (IQR Method)
    # Only apply to numerical columns
    numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns
    
    if len(numerical_cols) > 0:
        print(f"\nOutlier Detection & Handling (IQR Method):")
        
        outlier_counts = {}
        
        for col in numerical_cols:
            # Calculate IQR
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            
            # Define bounds
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Detect outliers
            outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
            outlier_count = len(outliers)
            
            if outlier_count > 0:
                outlier_counts[col] = outlier_count
                
                # Cap/floor outliers (Winsorization)
                df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
                print(f"  ✓ {col}: Capped {outlier_count} outliers")
        
        if outlier_counts:
            print(f"\nTotal outliers handled: {sum(outlier_counts.values())}")
        else:
            print("  ✓ No outliers detected")
    else:
        print("\n✓ No numerical columns to check for outliers")
    
    # Ensure categorical columns are category type
    object_cols = df_clean.select_dtypes(include=['object']).columns
    for col in object_cols:
        try:
            df_clean[col] = df_clean[col].astype('category')
            print(f"✓ Converted {col} to category type")
        except:
            pass
    
    # Display final summary
    print(f"\n{'─'*40}")
    print(f"Final Summary for {df_name}:")
    print(f"  Shape: {df_clean.shape}")
    print(f"  Data Types:\n{df_clean.dtypes}")
    print(f"  Memory Usage: {df_clean.memory_usage().sum() / 1024:.2f} KB")
    print(f"{'─'*40}")
    
    return df_clean

In [4]:
cleaned_dfs = {}

for df_name, df in dfs.items():
    cleaned_dfs[df_name] = clean_dataframe(df, df_name)


Processing: Crud_Oil_Price
✓ Converted Date to datetime

Missing Values Found:
Date    859
Vol.    218
dtype: int64
✓ Filled missing values in Date with mode
✓ Filled missing values in Vol. with mode

✓ No duplicate rows found

Outlier Detection & Handling (IQR Method):
  ✓ Price: Capped 230 outliers
  ✓ Open: Capped 234 outliers
  ✓ High: Capped 227 outliers
  ✓ Low: Capped 245 outliers

Total outliers handled: 936
✓ Converted Vol. to category type
✓ Converted Change % to category type

────────────────────────────────────────
Final Summary for Crud_Oil_Price:
  Shape: (1418, 7)
  Data Types:
Date        datetime64[ns]
Price              float64
Open               float64
High               float64
Low                float64
Vol.              category
Change %          category
dtype: object
  Memory Usage: 121.18 KB
────────────────────────────────────────

Processing: uk_cpih_fuel_energy_inflation
✓ Converted Time to datetime
✓ Converted mmm-yy to datetime

Missing Values Found:
Ti

C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\123045651.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\123045651.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\123045651.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\12

✓ Filled missing values in Time with mode
✓ Filled missing values in mmm-yy with mode
✓ Filled missing values in 07.1.1 New Cars with median
✓ Filled missing values in 07.1.1.1 New motor cars with median
✓ Filled missing values in 07.1.1.2 Second-hand motor cars with median
✓ Filled missing values in 07.1.1b Second Hand Cars with median
✓ Filled missing values in 07.1.2/3 Motocycles and bicycles with median
✓ Filled missing values in 07.1.2/3 Motorcycles and bicycles with median
✓ Filled missing values in 09.1.5 Repair of audio-visual equipment and related products with median
✓ Filled missing values in 09.1.5 Repair of audio-visual equipment, related products with median

✓ No duplicate rows found

Outlier Detection & Handling (IQR Method):
  ✓ 01.1.5 Oils and fats: Capped 30 outliers
  ✓ 01.1.9 Food products (nec): Capped 37 outliers
  ✓ 01.2 Non-alcoholic beverages: Capped 8 outliers
  ✓ 01.2.2 Mineral waters, soft drinks and juices: Capped 12 outliers
  ✓ 02.1.3 Beer: Capped 55 out

C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\123045651.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
C:\Users\touhi\AppData\Local\Temp\ipykernel_14488\123045651.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')


✓ Filled missing values in 09.6 Package holidays_YoY with median
✓ Filled missing values in 10 Education_YoY with median
✓ Filled missing values in 11 Restaurants and hotels_YoY with median
✓ Filled missing values in 11.1 Catering services_YoY with median
✓ Filled missing values in 11.1.1 Restaurants and cafes_YoY with median
✓ Filled missing values in 11.1.2 Canteens_YoY with median
✓ Filled missing values in 11.2 Accommodation services_YoY with median
✓ Filled missing values in 12 Miscellaneous goods and services_YoY with median
✓ Filled missing values in 12.1 Personal care_YoY with median
✓ Filled missing values in 12.1.1 Hairdressing and personal grooming establishments_YoY with median
✓ Filled missing values in 12.1.2/3 Appliances and products for personal care_YoY with median
✓ Filled missing values in 12.3 Personal effects (nec)_YoY with median
✓ Filled missing values in 12.3.1 Jewellery, clocks and watches_YoY with median
✓ Filled missing values in 12.3.2 Other personal effects

c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract


✓ No duplicate rows found

Outlier Detection & Handling (IQR Method):
  ✓ Value: Capped 3 outliers

Total outliers handled: 3
✓ Converted Category to category type
✓ Converted Income_Group to category type

────────────────────────────────────────
Final Summary for uk_fuel_summary_by_income_group:
  Shape: (72, 4)
  Data Types:
Year               int64
Category        category
Income_Group    category
Value            float64
dtype: object
  Memory Usage: 1.72 KB
────────────────────────────────────────

Processing: uk_fuel_transport_spending_by_decile

✓ No missing values found

✓ No duplicate rows found

Outlier Detection & Handling (IQR Method):
  ✓ Value: Capped 24 outliers

Total outliers handled: 24
✓ Converted Category to category type
✓ Converted Decile to category type

────────────────────────────────────────
Final Summary for uk_fuel_transport_spending_by_decile:
  Shape: (420, 4)
  Data Types:
Year           int64
Category    category
Decile      category
Value        floa

In [5]:
#  Save cleaned datasets to processed folder

processed_path = '../data/processed/'
os.makedirs(processed_path, exist_ok=True)

for df_name, df_clean in cleaned_dfs.items():
    # Convert category columns back to object for CSV saving
    for col in df_clean.select_dtypes(include=['category']).columns:
        df_clean[col] = df_clean[col].astype(str)
    
    # Convert datetime columns to string for CSV
    for col in df_clean.select_dtypes(include=['datetime64']).columns:
        df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d')
    
    # Save to CSV
    output_file = os.path.join(processed_path, f"{df_name}_clean.csv")
    df_clean.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")


✓ Saved: ../data/processed/Crud_Oil_Price_clean.csv
✓ Saved: ../data/processed/uk_cpih_fuel_energy_inflation_clean.csv
✓ Saved: ../data/processed/uk_cpih_inflation_by_category_clean.csv
✓ Saved: ../data/processed/uk_cpih_inflation_with_yoy_clean.csv
✓ Saved: ../data/processed/uk_fuel_summary_by_income_group_clean.csv
✓ Saved: ../data/processed/uk_fuel_transport_spending_by_decile_clean.csv
✓ Saved: ../data/processed/uk_household_expenditure_by_decile_clean.csv
✓ Saved: ../data/processed/uk_household_spending_by_income_group_clean.csv
✓ Saved: ../data/processed/uk_petrol_diesel_clean.csv
✓ Saved: ../data/processed/uk_petrol_diesel_spending_by_decile_clean.csv


In [6]:
# Verify all cleaned files
print(f"\n{'='*60}")
print("VERIFICATION - All Cleaned Datasets")
print(f"{'='*60}")

for df_name, df_clean in cleaned_dfs.items():
    print(f"\n{df_name}:")
    print(f"  Shape: {df_clean.shape}")
    print(f"  Null values: {df_clean.isnull().sum().sum()}")
    print(f"  Duplicates: {df_clean.duplicated().sum()}")


VERIFICATION - All Cleaned Datasets

Crud_Oil_Price:
  Shape: (1418, 7)
  Null values: 0
  Duplicates: 1

uk_cpih_fuel_energy_inflation:
  Shape: (457, 7)
  Null values: 0
  Duplicates: 0

uk_cpih_inflation_by_category:
  Shape: (457, 128)
  Null values: 0
  Duplicates: 0

uk_cpih_inflation_with_yoy:
  Shape: (457, 254)
  Null values: 0
  Duplicates: 0

uk_fuel_summary_by_income_group:
  Shape: (72, 4)
  Null values: 0
  Duplicates: 0

uk_fuel_transport_spending_by_decile:
  Shape: (420, 4)
  Null values: 0
  Duplicates: 0

uk_household_expenditure_by_decile:
  Shape: (720, 4)
  Null values: 0
  Duplicates: 0

uk_household_spending_by_income_group:
  Shape: (1440, 4)
  Null values: 0
  Duplicates: 0

uk_petrol_diesel:
  Shape: (2049, 3)
  Null values: 0
  Duplicates: 0

uk_petrol_diesel_spending_by_decile:
  Shape: (180, 4)
  Null values: 0
  Duplicates: 0


In [3]:
import os
# Read all files from processed folder and display information
processed_path = '../data/processed/'
# Get all CSV files
processed_files = [f for f in os.listdir(processed_path) if f.endswith('.csv')]

In [6]:
# Dictionary to store dataframes
processed_dfs = {}

print("="*80)
print("READING ALL PROCESSED FILES")
print("="*80)

for file in processed_files:
    print(f"\n{'='*60}")
    print(f"FILE: {file}")
    print(f"{'='*60}")
    
    # Read CSV
    df = pd.read_csv(os.path.join(processed_path, file))
    processed_dfs[file.replace('_clean.csv', '')] = df
    
    # Display information
    print(f"\nShape: {df.shape}")
    print(f"\nColumn Names:")
    print(df.columns.tolist())
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nFirst 10 rows:")
    print(df.head(10))
    print(f"\nMissing Values:")
    print(df.isnull().sum())
    print(f"\nDescriptive Statistics (numerical columns):")
    print(df.describe())
    print("\n" + "-"*60)


READING ALL PROCESSED FILES

FILE: Crud_Oil_Price_clean.csv

Shape: (1418, 7)

Column Names:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

Data Types:
Date         object
Price       float64
Open        float64
High        float64
Low         float64
Vol.         object
Change %     object
dtype: object

First 10 rows:
         Date   Price       Open       High        Low    Vol. Change %
0  2026-10-08  727.55  726.41625  732.15625  721.02875   0.00K    2.14%
1  2026-07-08  727.55  726.41625  732.15625  721.02875   2.50K    2.27%
2  2026-06-08  727.55  726.41625  732.15625  721.02875   3.09K    2.26%
3  2026-05-08  727.55  726.41625  732.15625  721.02875   0.44K   -0.62%
4  2026-04-08  727.55  726.41625  732.15625  721.02875   0.70K   -3.90%
5  2026-03-08  727.55  726.41625  732.15625  721.02875   0.27K   -3.27%
6  2021-01-02  727.55  726.41625  732.15625  721.02875   1.60K    0.94%
7  2021-01-02  727.55  726.41625  732.15625  721.02875  14.81K   -1.33%
8  2021-01-02  

c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeW

In [7]:
#  Create summary of all files
print("\n" + "="*80)
print("SUMMARY OF ALL PROCESSED FILES")
print("="*80)

summary_df = pd.DataFrame({
    'File_Name': [f.replace('_clean.csv', '') for f in processed_files],
    'Shape': [df.shape for df in processed_dfs.values()],
    'Columns': [df.columns.tolist() for df in processed_dfs.values()],
    'Numeric_Columns': [df.select_dtypes(include=['float64', 'int64']).columns.tolist() for df in processed_dfs.values()],
    'Date_Columns': [[col for col in df.columns if 'Date' in col or 'Time' in col or col == 'mmm-yy'] for df in processed_dfs.values()],
    'Total_Null': [df.isnull().sum().sum() for df in processed_dfs.values()],
    'Duplicates': [df.duplicated().sum() for df in processed_dfs.values()]
})

print(summary_df.to_string())


SUMMARY OF ALL PROCESSED FILES
                               File_Name       Shape                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    